# 🤖 ML Training Pipeline - Master Orchestrator

**Purpose:** Orchestrates the complete machine learning training pipeline for fantasy football player performance prediction. Runs weekly after all fresh data is available.

---

## 📊 Pipeline Architecture

```
┌──────────────────────────────────────────────┐
│   00_ML_Training_Master_Orchestrator         │
│             (This Notebook)                  │
└──────────────┬───────────────────────────────┘
               │
       ┌───────┼───────┐
       │       │       │
       ▼       ▼       ▼
┌──────────┐ ┌────┐ ┌──────┐
│ Feature  │ │Data│ │Model │
│Engineer- │→│Prep│→│Train-│
│   ing    │ │    │ │ ing  │
└──────────┘ └────┘ └──────┘
```

---

## 📦 Pipeline Steps

| Step | Notebook | Output | Estimated Duration |
| --- | --- | --- | --- |
| **1. Feature Engineering** | ml_feature_engineering | Feature tables | ~5 min |
| **2. Data Preparation** | ml_prediction_data_prep | Train/test splits | ~3 min |
| **3. Model Training** | ml_prediction_model_training | Trained models + UC registration | ~20 min |

**Total Pipeline:** ~30 minutes

---

## ⏰ Schedule

**Weekly: Tuesdays at 5:00 AM CT**
* Runs after all weekly data jobs complete (3 AM CT)
* Fresh stats from Monday Night Football available
* Models ready for upcoming week's predictions

In [0]:
from datetime import datetime
import traceback

print("="*80)
print("🤖 ML TRAINING PIPELINE - MASTER ORCHESTRATOR")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# Pipeline configuration
base_path = "/Repos/kingoffrisco@yahoo.com/FantasAI/notebooks/03_ML_Training"
pipeline_steps = [
    {
        "name": "Feature Engineering",
        "notebook": f"{base_path}/ml_feature_engineering",
        "timeout": 600  # 10 minutes
    },
    {
        "name": "Data Preparation",
        "notebook": f"{base_path}/ml_prediction_data_prep",
        "timeout": 300  # 5 minutes
    },
    {
        "name": "Model Training",
        "notebook": f"{base_path}/ml_prediction_model_training",
        "timeout": 1800  # 30 minutes
    }
]

# Track results
pipeline_results = []
pipeline_start = datetime.now()

print(f"\n📋 Pipeline Steps: {len(pipeline_steps)}")
for i, step in enumerate(pipeline_steps, 1):
    print(f"   {i}. {step['name']}")

In [0]:
# Execute each step in sequence
for i, step in enumerate(pipeline_steps, 1):
    step_name = step['name']
    notebook_path = step['notebook']
    timeout = step['timeout']
    
    print("\n" + "="*80)
    print(f"📌 STEP {i}/{len(pipeline_steps)}: {step_name}")
    print(f"   Notebook: {notebook_path}")
    print(f"   Timeout: {timeout}s")
    print("="*80)
    
    step_start = datetime.now()
    
    try:
        # Run the notebook
        result = dbutils.notebook.run(
            notebook_path,
            timeout_seconds=timeout
        )
        step_end = datetime.now()
        duration = (step_end - step_start).total_seconds()

        # Minimal fix: handle missing output
        if not result or result == "" or "NotebookExecutionException" in str(result):
            print(f"\n❌ {step_name} FAILED - No output returned or notebook exception")
            pipeline_results.append({
                "step": i,
                "name": step_name,
                "status": "FAILED",
                "duration": duration,
                "result": None,
                "error": "No output returned or NotebookExecutionException"
            })
            print("\n⚠️  Pipeline halted due to failure")
            break
        print(f"\n✅ {step_name} completed successfully")
        print(f"   Duration: {duration:.1f}s")
        print(f"   Result: {result}")
        pipeline_results.append({
            "step": i,
            "name": step_name,
            "status": "SUCCESS",
            "duration": duration,
            "result": result,
            "error": None
        })
        
    except Exception as e:
        step_end = datetime.now()
        duration = (step_end - step_start).total_seconds()
        error_msg = str(e)
        error_trace = traceback.format_exc()
        
        print(f"\n❌ {step_name} FAILED")
        print(f"   Duration: {duration:.1f}s")
        print(f"   Error: {error_msg}")
        print(f"\n{error_trace}")
        pipeline_results.append({
            "step": i,
            "name": step_name,
            "status": "FAILED",
            "duration": duration,
            "result": None,
            "error": error_msg
        })
        print("\n⚠️  Pipeline halted due to failure")
        break

In [0]:
# Calculate totals
pipeline_end = datetime.now()
total_duration = (pipeline_end - pipeline_start).total_seconds()

successful_steps = sum(1 for r in pipeline_results if r['status'] == 'SUCCESS')
failed_steps = sum(1 for r in pipeline_results if r['status'] == 'FAILED')

print("\n" + "="*80)
print("📊 PIPELINE SUMMARY")
print("="*80)
print(f"Started:  {pipeline_start.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {pipeline_end.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total Duration: {total_duration:.1f}s ({total_duration/60:.1f} min)")
print(f"\nSteps: {len(pipeline_steps)} total | {successful_steps} ✅ | {failed_steps} ❌")

print("\n" + "-"*80)
for result in pipeline_results:
    status_icon = "✅" if result['status'] == 'SUCCESS' else "❌"
    print(f"{status_icon} Step {result['step']}: {result['name']}")
    print(f"   Duration: {result['duration']:.1f}s")
    if result['error']:
        print(f"   Error: {result['error']}")
print("-"*80)

# Overall status
if failed_steps == 0:
    print("\n✅ PIPELINE COMPLETED SUCCESSFULLY")
    dbutils.notebook.exit("SUCCESS")
else:
    print(f"\n❌ PIPELINE FAILED - {failed_steps} step(s) failed")
    raise Exception(f"Pipeline failed: {failed_steps} of {len(pipeline_steps)} steps failed")